# 05 - 全量召回率优先 vs 精度优先（time + sid）

本 Notebook 在**全量数据**上分别跑两套工程向流水线，**模型与日志独立保存**（与 `04_run_pipelines.ipynb` 的 F1 最大化实验互不覆盖）。

| 策略 | marker 前缀 | 实验 YAML | run_name（产物目录） | 阈值策略 |
|------|-------------|-----------|----------------------|----------|
| **召回率优先** | `full_recall` / `full_recall_sid` | `exp_full_recall_*.yaml` | `runs/exp_tianchi_full_recall_*` | `threshold_profile: recall_guard` |
| **精度优先** | `full_precision` / `full_precision_sid` | `exp_full_precision_*.yaml` | `runs/exp_tianchi_full_precision_*` | `threshold_profile: precision_guard` |

**共享 preprocess**：两套策略使用相同 data 配置（`drop_empty_positive_windows: true`），仅跑 **一次** preprocess（marker `full_eng__preprocess`）。

**日志**：`runs/pipeline-logs/{marker}__{step}__{timestamp}.log`  
**模型**：`runs/exp_tianchi_full_recall_ddp/`、`runs/exp_tianchi_full_precision_ddp/` 等（各自 `.latest_train_session` / `.latest_infer_session`）。

项目 KPI（F1 最大化）见 **`04_run_pipelines.ipynb`**。

**Notebook 提示**：修改 Python 源码后请先重跑 setup 单元，或 **Kernel → Restart**。

In [1]:
import os
from pathlib import Path

from dram2cnn.utils.notebook_bootstrap import notebook_setup
from dram2cnn.utils.pipeline_helpers import pipeline_log_dir, pipeline_marker_dir

PROJECT_ROOT = notebook_setup(cwd=Path.cwd().resolve())
LOG_DIR = pipeline_log_dir(PROJECT_ROOT)
MARKER_DIR = pipeline_marker_dir(PROJECT_ROOT)

print("PROJECT_ROOT:", PROJECT_ROOT)


PROJECT_ROOT: /root/workspace


In [2]:
from dram2cnn.utils.config import load_experiment_config
from dram2cnn.utils.pipeline_helpers import (
    clear_exp_markers,
    preflight_full_scale_index,
    preflight_split_labels,
    preflight_splits_vs_model,
    processed_dir_for_exp,
    require_preprocess_manifest,
    run_step,
    splits_dir_for_exp,
)
from dram2cnn.utils.deployment_kpi import (
    RECALL_PRECISION_KPI_PAIRS,
    collect_kpi_rows,
    merge_all_strategies_kpi,
    write_kpi_csv,
)

# 实验 YAML / marker 前缀（全 notebook 共用）
ENG_EXP = "full_eng"
ENG_PREPROCESS_CFG = "configs/experiments/exp_full_recall_ddp.yaml"
RECALL_TIME_EXP = "full_recall"
RECALL_TIME_CFG = "configs/experiments/exp_full_recall_ddp.yaml"
RECALL_SID_EXP = "full_recall_sid"
RECALL_SID_CFG = "configs/experiments/exp_full_recall_sid.yaml"
PREC_TIME_EXP = "full_precision"
PREC_TIME_CFG = "configs/experiments/exp_full_precision_ddp.yaml"
PREC_SID_EXP = "full_precision_sid"
PREC_SID_CFG = "configs/experiments/exp_full_precision_sid.yaml"

print("pipeline helpers loaded (run_step, preflight, deployment_kpi, 实验常量)")

pipeline helpers loaded (run_step, preflight, deployment_kpi, 实验常量)


---
## 共享：全量索引 + 工程向 preprocess

使用 `exp_full_recall_ddp.yaml` 的 data 段（与 precision 实验一致：`drop_empty_positive_windows: true`）。

> 若已用 **04（F1）** 跑过 preprocess（`drop_empty_positive_windows: false`），请先 `rm -rf data/processed/* data/splits/*` 并 `clear_exp_markers('full_eng')`。

In [3]:
_cfg = preflight_full_scale_index(ENG_PREPROCESS_CFG, root=PROJECT_ROOT, context="05_eng_preflight")

run_step(ENG_EXP, "build_index", "bash scripts/build_index.sh full")
run_step(ENG_EXP, "preprocess", f"bash scripts/preprocess.sh {ENG_PREPROCESS_CFG}")
run_step(ENG_EXP, "inspect_data", f"bash scripts/inspect_data.sh {ENG_PREPROCESS_CFG}")

_pd = processed_dir_for_exp(ENG_PREPROCESS_CFG, PROJECT_ROOT)
_sd = splits_dir_for_exp(ENG_PREPROCESS_CFG, PROJECT_ROOT)
print(f"工程向 preprocess 完成 → {_pd.relative_to(PROJECT_ROOT)}/ + {_sd.relative_to(PROJECT_ROOT)}/")

[RUN ] full_eng/build_index
      command: bash scripts/build_index.sh full
      log: /root/workspace/runs/pipeline-logs/full_eng__build_index__20260617_063235.log
Building index mode=full horizon=300.0s interval=300.0s window=300.0s pos_interval=60.0s → /root/workspace/data/raw/features_index_h300s_step300s_full.csv
Done: rows=66320 pos=9948 neg=56372 pos_rate=0.1500
[DONE] full_eng/build_index

[RUN ] full_eng/preprocess
      command: bash scripts/preprocess.sh configs/experiments/exp_full_recall_ddp.yaml
      log: /root/workspace/runs/pipeline-logs/full_eng__preprocess__20260617_063724.log
Scanning mcelog for row/col max …
Grid representation='spatial_heatmap', (H,W)=(128,32), channels=31
Computing global time epoch offset (earliest raw timestamp → relative t=0) …
  time_epoch_offset_absolute=-62135596800.000
Loading index CSV …
Loading inventory one-hot …
预处理并行度: worker 预算=1（CPU 上限=67，floor(112×0.6)，RAM 预算=31.2GiB（62GiB×0.5，目标≤50%），估算峰值≈38.0GiB，已由 67 压至 1，警告：即使最小 worker 估算峰值仍超预算

---
## 召回率优先（宁报勿漏）

- **time**：`exp_full_recall_ddp.yaml` → `runs/exp_tianchi_full_recall_ddp/`
- **sid**：需 resplit → `runs/exp_tianchi_full_recall_sid/`
- 阈值：`recall_guard`（recall ≥ 0.88 且 precision ≥ 下限）

In [4]:
_cfg = load_experiment_config(PROJECT_ROOT / RECALL_TIME_CFG)
print("threshold_profile:", _cfg["train"].get("threshold_profile"))
print("threshold_search:", _cfg["train"].get("threshold_search"))

preflight_splits_vs_model(RECALL_TIME_CFG)
preflight_split_labels(RECALL_TIME_CFG)

run_step(RECALL_TIME_EXP, "train", f"bash scripts/train_ddp.sh {RECALL_TIME_CFG}")
run_step(RECALL_TIME_EXP, "infer", f"bash scripts/infer.sh {RECALL_TIME_CFG}")
run_step(RECALL_TIME_EXP, "evaluate", f"bash scripts/evaluate.sh {RECALL_TIME_CFG}")

print("召回率优先 time 完成 → runs/exp_tianchi_full_recall_ddp/")

threshold_profile: recall_guard
threshold_search: {'objective': 'min_recall', 'min_recall': 0.88, 'min_precision': 0.55, 'min': 0.05, 'max': 0.95, 'step': 0.01, 'enabled': True, 'f_beta': 2.0}
[preflight] 通过 (configs/experiments/exp_full_recall_ddp.yaml): X_train (C,H,W)=(31,128,32) splits=/root/workspace/data/splits
[preflight] split_manifest (split_manifest.json): time_stratified
[preflight] configs/experiments/exp_full_recall_ddp.yaml train: N=21742 pos=3747 rate=0.1723
[preflight] configs/experiments/exp_full_recall_ddp.yaml val: N=4659 pos=804 rate=0.1726
[preflight] configs/experiments/exp_full_recall_ddp.yaml test: N=4659 pos=803 rate=0.1724
[RUN ] full_recall/train
      command: bash scripts/train_ddp.sh configs/experiments/exp_full_recall_ddp.yaml
      log: /root/workspace/runs/pipeline-logs/full_recall__train__20260617_065704.log
[train_ddp.sh] CUDA_VISIBLE_DEVICES=<not-set, use all visible GPUs>
[train_ddp.sh] exp_config=configs/experiments/exp_full_recall_ddp.yaml
[train_

In [5]:
require_preprocess_manifest(
    ENG_PREPROCESS_CFG,
    root=PROJECT_ROOT,
    hint_cmd=f"bash scripts/preprocess.sh {ENG_PREPROCESS_CFG}",
)
run_step(RECALL_SID_EXP, "resplit", f"bash scripts/resplit.sh {RECALL_SID_CFG}")
preflight_splits_vs_model(RECALL_SID_CFG)
preflight_split_labels(RECALL_SID_CFG)

run_step(RECALL_SID_EXP, "train", f"bash scripts/train_ddp.sh {RECALL_SID_CFG}")
run_step(RECALL_SID_EXP, "infer", f"bash scripts/infer.sh {RECALL_SID_CFG}")
run_step(RECALL_SID_EXP, "evaluate", f"bash scripts/evaluate.sh {RECALL_SID_CFG}")

print("召回率优先 sid 完成 → runs/exp_tianchi_full_recall_sid/")

[RUN ] full_recall_sid/resplit
      command: bash scripts/resplit.sh configs/experiments/exp_full_recall_sid.yaml
      log: /root/workspace/runs/pipeline-logs/full_recall_sid__resplit__20260617_070717.log
Split-only: 复用 processed/（n_samples=31060），策略=sid
Preprocess + split completed. [31, 128, 32]
Validate after preprocess: ok
[DONE] full_recall_sid/resplit

[preflight] 通过 (configs/experiments/exp_full_recall_sid.yaml): X_train (C,H,W)=(31,128,32) splits=/root/workspace/data/splits_sid
[preflight] split_manifest (split_manifest.json): sid
[preflight] configs/experiments/exp_full_recall_sid.yaml train: N=21713 pos=3745 rate=0.1725
[preflight] configs/experiments/exp_full_recall_sid.yaml val: N=4689 pos=813 rate=0.1734
[preflight] configs/experiments/exp_full_recall_sid.yaml test: N=4658 pos=796 rate=0.1709
[RUN ] full_recall_sid/train
      command: bash scripts/train_ddp.sh configs/experiments/exp_full_recall_sid.yaml
      log: /root/workspace/runs/pipeline-logs/full_recall_sid__tra

---
## 精度优先（高精度告警）

- **time**：`exp_full_precision_ddp.yaml` → `runs/exp_tianchi_full_precision_ddp/`
- **sid**：`exp_full_precision_sid.yaml` → `runs/exp_tianchi_full_precision_sid/`
- 阈值：`precision_guard`（precision ≥ 0.65 下最大化 recall）

In [6]:
_cfg = load_experiment_config(PROJECT_ROOT / PREC_TIME_CFG)
print("threshold_profile:", _cfg["train"].get("threshold_profile"))

preflight_splits_vs_model(PREC_TIME_CFG)
preflight_split_labels(PREC_TIME_CFG)

run_step(PREC_TIME_EXP, "train", f"bash scripts/train_ddp.sh {PREC_TIME_CFG}")
run_step(PREC_TIME_EXP, "infer", f"bash scripts/infer.sh {PREC_TIME_CFG}")
run_step(PREC_TIME_EXP, "evaluate", f"bash scripts/evaluate.sh {PREC_TIME_CFG}")

print("精度优先 time 完成 → runs/exp_tianchi_full_precision_ddp/")

threshold_profile: precision_guard
[preflight] 通过 (configs/experiments/exp_full_precision_ddp.yaml): X_train (C,H,W)=(31,128,32) splits=/root/workspace/data/splits
[preflight] split_manifest (split_manifest.json): time_stratified
[preflight] configs/experiments/exp_full_precision_ddp.yaml train: N=21742 pos=3747 rate=0.1723
[preflight] configs/experiments/exp_full_precision_ddp.yaml val: N=4659 pos=804 rate=0.1726
[preflight] configs/experiments/exp_full_precision_ddp.yaml test: N=4659 pos=803 rate=0.1724
[RUN ] full_precision/train
      command: bash scripts/train_ddp.sh configs/experiments/exp_full_precision_ddp.yaml
      log: /root/workspace/runs/pipeline-logs/full_precision__train__20260617_073044.log
[train_ddp.sh] CUDA_VISIBLE_DEVICES=<not-set, use all visible GPUs>
[train_ddp.sh] exp_config=configs/experiments/exp_full_precision_ddp.yaml
[train_ddp.sh] index_features_path OK: /root/workspace/data/raw/features_index_h300s_step300s_full.csv
[gpu_launch:train] visible GPUs (3): c

In [7]:
require_preprocess_manifest(
    ENG_PREPROCESS_CFG,
    root=PROJECT_ROOT,
    hint_cmd=f"bash scripts/preprocess.sh {ENG_PREPROCESS_CFG}",
)
run_step(PREC_SID_EXP, "resplit", f"bash scripts/resplit.sh {PREC_SID_CFG}")
preflight_splits_vs_model(PREC_SID_CFG)
preflight_split_labels(PREC_SID_CFG)

run_step(PREC_SID_EXP, "train", f"bash scripts/train_ddp.sh {PREC_SID_CFG}")
run_step(PREC_SID_EXP, "infer", f"bash scripts/infer.sh {PREC_SID_CFG}")
run_step(PREC_SID_EXP, "evaluate", f"bash scripts/evaluate.sh {PREC_SID_CFG}")

print("精度优先 sid 完成 → runs/exp_tianchi_full_precision_sid/")

[RUN ] full_precision_sid/resplit
      command: bash scripts/resplit.sh configs/experiments/exp_full_precision_sid.yaml
      log: /root/workspace/runs/pipeline-logs/full_precision_sid__resplit__20260617_074021.log
Split-only: 复用 processed/（n_samples=31060），策略=sid
Preprocess + split completed. [31, 128, 32]
Validate after preprocess: ok
[DONE] full_precision_sid/resplit

[preflight] 通过 (configs/experiments/exp_full_precision_sid.yaml): X_train (C,H,W)=(31,128,32) splits=/root/workspace/data/splits_sid
[preflight] split_manifest (split_manifest.json): sid
[preflight] configs/experiments/exp_full_precision_sid.yaml train: N=21713 pos=3745 rate=0.1725
[preflight] configs/experiments/exp_full_precision_sid.yaml val: N=4689 pos=813 rate=0.1734
[preflight] configs/experiments/exp_full_precision_sid.yaml test: N=4658 pos=796 rate=0.1709
[RUN ] full_precision_sid/train
      command: bash scripts/train_ddp.sh configs/experiments/exp_full_precision_sid.yaml
      log: /root/workspace/runs/pipe

In [8]:
# 召回 vs 精度：四路 deployment KPI 汇总
rows = collect_kpi_rows(RECALL_PRECISION_KPI_PAIRS, project_root_path=PROJECT_ROOT)
if rows:
    import pandas as pd

    cmp_df = pd.DataFrame(rows)
    print(cmp_df.to_string(index=False))
    out = PROJECT_ROOT / "outputs" / "tables"
    write_kpi_csv(rows, out / "full_recall_vs_precision_deployment_kpi.csv")
    merged = merge_all_strategies_kpi(tables_dir=out)
    print("\n已写入 outputs/tables/full_recall_vs_precision_deployment_kpi.csv")
    if merged:
        print(f"已更新汇总表 {merged.name}")

      strategy                       run_name  threshold       f1  precision   recall  roc_auc                                                                                  metrics_path
   recall/time    exp_tianchi_full_recall_ddp       0.23 0.382748   0.241447 0.922790 0.764328    /root/workspace/runs/exp_tianchi_full_recall_ddp/infer_20260617_070706_516837/metrics.json
    recall/sid    exp_tianchi_full_recall_sid       0.10 0.437519   0.287288 0.917085 0.822231    /root/workspace/runs/exp_tianchi_full_recall_sid/infer_20260617_073032_900962/metrics.json
precision/time exp_tianchi_full_precision_ddp       0.81 0.432812   0.580713 0.344956 0.769937 /root/workspace/runs/exp_tianchi_full_precision_ddp/infer_20260617_074010_251426/metrics.json
 precision/sid exp_tianchi_full_precision_sid       0.79 0.414023   0.616915 0.311558 0.820360 /root/workspace/runs/exp_tianchi_full_precision_sid/infer_20260617_080250_485098/metrics.json

已写入 outputs/tables/full_recall_vs_precision_deployment

In [9]:
print("=== 本 Notebook 相关 marker ===")
for prefix in ("full_eng", "full_recall", "full_recall_sid", "full_precision", "full_precision_sid"):
    for p in sorted(MARKER_DIR.glob(f"{prefix}__*.done")):
        print(p.name, "=>", p.read_text(encoding="utf-8").strip())

=== 本 Notebook 相关 marker ===
full_eng__build_index.done => 20260617_063235
full_eng__inspect_data.done => 20260617_065701
full_eng__preprocess.done => 20260617_063724
full_recall__evaluate.done => 20260617_070713
full_recall__infer.done => 20260617_070648
full_recall__train.done => 20260617_065704
full_recall_sid__evaluate.done => 20260617_073039
full_recall_sid__infer.done => 20260617_073015
full_recall_sid__resplit.done => 20260617_070717
full_recall_sid__train.done => 20260617_071839
full_precision__evaluate.done => 20260617_074017
full_precision__infer.done => 20260617_073951
full_precision__train.done => 20260617_073044
full_precision_sid__evaluate.done => 20260617_080256
full_precision_sid__infer.done => 20260617_080232
full_precision_sid__resplit.done => 20260617_074021
full_precision_sid__train.done => 20260617_075210


In [10]:
# 重置 marker（可选）
# clear_exp_markers("full_eng")
# clear_exp_markers("full_recall")
# clear_exp_markers("full_recall_sid")
# clear_exp_markers("full_precision")
# clear_exp_markers("full_precision_sid")
print("未删除 marker。")

未删除 marker。
